# Data Filter — Lọc cặp QA chất lượng thấp (Lịch sử Đảng)

Notebook này đọc `qa_audit.jsonl` từ `data/training_data/`, lọc cặp QA kém chất lượng, chia lại train/val/test và **ghi đè trực tiếp** các file JSON trong cùng folder.

## Vấn đề đang giải quyết

Dataset hiện tại có 2 vấn đề:
1. **Val = 0** — toàn bộ 1458 pairs gom hết vào train, không có tập đánh giá để chọn best checkpoint
2. **Query meta** — nhiều query hỏi về cấu trúc tài liệu thay vì nội dung kiến thức:
   - *"Chương 5 đề cập đến nội dung gì?"*
   - *"Tài liệu này nói về điều gì?"*
   - *"Đoạn văn trên đề cập đến ai?"*

## Pipeline lọc

```
qa_audit.jsonl  →  Hard filters (regex + heuristic)  →  Re-split 80/10/10  →  ghi đè JSON gốc
```

## Output (ghi vào `data/training_data/`)

| File | Nội dung |
|---|---|
| `qa_audit.jsonl` | Toàn bộ pairs sau lọc |
| `train_data.json` + `train_pairs_mnrl.jsonl` + `train_triplets.jsonl` | Train set 80% |
| `val_data.json` + `val_data_mnrl.jsonl` | Val set 10% |
| `test_data.json` + `test_data_mnrl.jsonl` | Test set 10% |
| `stats.json` | Thống kê sau lọc |
| `filter_report.json` | Chi tiết từng loại bị loại |

In [2]:
import re
import json
import random
import hashlib
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Dict, Tuple

# ── CONFIG ──────────────────────────────────────────────────────────────────
DATA_ROOT = Path("../data")

# Tất cả 3 thư mục cần xử lý
# training_data_th bị lồng thêm 1 cấp — trỏ thẳng vào thư mục chứa qa_audit.jsonl
def _resolve_subject_dir(candidate: Path) -> Path:
    """Nếu qa_audit.jsonl nằm trong thư mục con cùng tên, trả về thư mục con đó."""
    if not (candidate / "qa_audit.jsonl").is_file():
        nested = candidate / candidate.name
        if (nested / "qa_audit.jsonl").is_file():
            return nested
    return candidate

SUBJECT_DIRS = [
    _resolve_subject_dir(DATA_ROOT / "training_data"),
    _resolve_subject_dir(DATA_ROOT / "training_data_pldc"),
    _resolve_subject_dir(DATA_ROOT / "training_data_th"),
]

# Dùng thư mục đầu tiên để preview trong các cell kiểm tra (Bước 1-4)
SUBJECT_DIR = SUBJECT_DIRS[0]
AUDIT_FILE  = SUBJECT_DIR / "qa_audit.jsonl"

# Thresholds lọc
MIN_QUERY_CHARS        = 25
MAX_QUERY_CHARS        = 280
MIN_POSITIVE_OVERLAP   = 0.10   # tỉ lệ token query xuất hiện trong positive
ENABLE_SEMANTIC_FILTER = False  # bật True để lọc thêm bằng embedding (cần GPU)
SEMANTIC_THRESHOLD     = 0.35

# Re-split (sửa ở đây nếu muốn tỉ lệ khác)
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
# TEST_RATIO  = 1 - TRAIN_RATIO - VAL_RATIO = 0.10

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

print("Các thư mục sẽ xử lý:")
for i, d in enumerate(SUBJECT_DIRS, 1):
    audit = d / "qa_audit.jsonl"
    status = "✓" if audit.is_file() else "✗ KHÔNG TÌM THẤY"
    print(f"  [{i}] {d.resolve()}  {status}")
print()
print(f"Preview (Bước 1-4) : {AUDIT_FILE.resolve()}")
print(f"Min query chars    : {MIN_QUERY_CHARS}")
print(f"Split              : train={TRAIN_RATIO:.0%}  val={VAL_RATIO:.0%}  test={1-TRAIN_RATIO-VAL_RATIO:.0%}")
print(f"Semantic filter    : {ENABLE_SEMANTIC_FILTER}")

if not AUDIT_FILE.is_file():
    raise FileNotFoundError(f"Không tìm thấy: {AUDIT_FILE}")

Các thư mục sẽ xử lý:
  [1] C:\CS431\UniPolis\data\training_data  ✓
  [2] C:\CS431\UniPolis\data\training_data_pldc  ✓
  [3] C:\CS431\UniPolis\data\training_data_th\training_data_th  ✓

Preview (Bước 1-4) : C:\CS431\UniPolis\data\training_data\qa_audit.jsonl
Min query chars    : 25
Split              : train=80%  val=10%  test=10%
Semantic filter    : False


## Bước 1: Load dữ liệu

In [3]:
def load_jsonl(path: Path) -> List[Dict]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


all_pairs: List[Dict] = load_jsonl(AUDIT_FILE)
print(f"Đã load: {len(all_pairs)} pairs từ {AUDIT_FILE}")
print()

# Phân bổ intent ban đầu
intent_dist = Counter(p["intent_type"] for p in all_pairs)
print("Intent distribution (ban đầu):")
for k, v in sorted(intent_dist.items()):
    print(f"  {k:<15}: {v:>5}  ({v/len(all_pairs)*100:.1f}%)")

chapter_dist = Counter(str(p.get("chapter", "?"))[:60] for p in all_pairs)
print()
print("Chapter distribution:")
for chap, cnt in chapter_dist.most_common():
    print(f"  {cnt:>5}  {chap}")

Đã load: 1405 pairs từ ..\data\training_data\qa_audit.jsonl

Intent distribution (ban đầu):
  applicative    :   472  (33.6%)
  factual        :   452  (32.2%)
  relational     :   481  (34.2%)

Chapter distribution:
    628  Chương 3 
ĐẢNG LÃNH ĐẠO CẢ NƯỚC QUÁ ĐỘ LÊN
    394  Chương 2 
ĐẢNG LÃNH ĐẠO HAI CUỘC KHÁNG CHIẾN, HOÀN THÀNH GIẢ
    383  Chương 1 
ĐẢNG CỘNG SẢN VIỆT NAM RA ĐỜI VÀ LÃNH ĐẠO


## Bước 2: Hard filters (regex + heuristic)

Các loại cặp bị loại:

| Lý do | Mô tả |
|---|---|
| `meta_chapter` | Hỏi về chương/mục/phần (vd: "Chương 5 nói về gì?") |
| `meta_document` | Hỏi về tài liệu/giáo trình (vd: "Tài liệu này đề cập...") |
| `vague_pronoun` | Đại từ chỉ định mơ hồ ("đoạn này", "phần trên") |
| `meta_author` | Hỏi về tác giả/người biên soạn |
| `too_short` / `too_long` | Vượt ngưỡng độ dài |
| `no_words` | Không có từ tiếng Việt có nghĩa |
| `low_overlap` | Query không có token nào xuất hiện trong positive |
| `no_negatives` | Thiếu negatives (lỗi từ bước sinh) |

In [4]:
META_PATTERNS = {
    "meta_chapter": [
        re.compile(r"\b(chương|chuong)\s+\d+", re.IGNORECASE),
        re.compile(r"\b(mục|muc|phần|phan|bài|bai)\s+\d+", re.IGNORECASE),
    ],
    "meta_document": [
        re.compile(r"\b(tài liệu|giáo trình|sách|văn bản)\s+(này|đó|trên|dưới)", re.IGNORECASE),
        re.compile(r"\b(trong|theo)\s+(tài liệu|giáo trình|sách|nội dung|đoạn văn)", re.IGNORECASE),
        re.compile(r"^(tài liệu|giáo trình|sách|nội dung)\s", re.IGNORECASE),
    ],
    "vague_pronoun": [
        re.compile(r"\b(đoạn|phần|nội dung|mục)\s+(văn|trên|này|sau|dưới|đó|đầu|cuối)", re.IGNORECASE),
        re.compile(r"\btheo\s+(như\s+)?(trên|đoạn|tác giả|văn bản)", re.IGNORECASE),
        re.compile(r"\b(như\s+)?(đã\s+)?(nêu|trình bày|đề cập|nói)\s+(ở\s+)?trên", re.IGNORECASE),
        re.compile(r"\bđược\s+(đề cập|nêu|trình bày)\s+(ở\s+)?(trên|đâu|trong)", re.IGNORECASE),
    ],
    "meta_author": [
        re.compile(r"\b(tác giả|chủ biên|người biên soạn|biên tập)", re.IGNORECASE),
    ],
    "meta_structure": [
        re.compile(r"\b(số trang|trang số|tiêu đề|nhà xuất bản|nxb)\b", re.IGNORECASE),
        re.compile(r"\b(học phần|môn học)\s+(này|đó)", re.IGNORECASE),
    ],
}

_VN_WORD_RE  = re.compile(r"[a-zA-ZÀ-ỹ]{3,}")
_TOKENIZE_RE = re.compile(r"[^\wÀ-ỹ]+", re.UNICODE)


def normalize_tokens(text: str) -> set:
    """Tokenize thô bằng split theo non-word, lowercase."""
    return {t for t in _TOKENIZE_RE.split(text.lower()) if len(t) >= 3}


_VN_STOPWORDS = {
    "của", "và", "là", "các", "có", "được", "trong", "cho", "với", "này", "đó", "khi",
    "như", "để", "từ", "đã", "sẽ", "không", "một", "những", "sau", "trước", "khác",
    "hay", "thì", "mà", "nào", "vào", "ra", "lên", "xuống", "đến", "theo", "về",
    "nhằm", "qua", "giữa", "trên", "dưới", "bằng", "vẫn", "còn", "chỉ", "đang",
    "phải", "cũng", "nên", "vì", "do", "bởi", "hoặc", "nhưng", "hơn", "rất",
    "điều", "việc", "sự", "mỗi", "tất", "cả",
}


def query_positive_overlap(query: str, positive: str) -> float:
    """Tỉ lệ token (bỏ stopwords) của query xuất hiện trong positive."""
    q_tokens = normalize_tokens(query) - _VN_STOPWORDS
    if not q_tokens:
        return 0.0
    return len(q_tokens & normalize_tokens(positive)) / len(q_tokens)


def hard_filter(pair: Dict) -> Tuple[bool, str]:
    """Trả về (passed, reason). passed=True → giữ lại."""
    query = pair.get("query", "").strip()

    if not query:
        return False, "empty"
    if len(query) < MIN_QUERY_CHARS:
        return False, "too_short"
    if len(query) > MAX_QUERY_CHARS:
        return False, "too_long"
    if not _VN_WORD_RE.search(query):
        return False, "no_words"

    for category, patterns in META_PATTERNS.items():
        for pat in patterns:
            if pat.search(query):
                return False, category

    if not pair.get("negatives") or len(pair["negatives"]) < 1:
        return False, "no_negatives"

    positive = pair.get("positive", "")
    if query_positive_overlap(query, positive) < MIN_POSITIVE_OVERLAP:
        return False, "low_overlap"

    return True, "ok"


# ── Apply ────────────────────────────────────────────────────────────────────
kept: List[Dict]                 = []
removed: Dict[str, List[Dict]]  = defaultdict(list)

for pair in all_pairs:
    passed, reason = hard_filter(pair)
    if passed:
        kept.append(pair)
    else:
        removed[reason].append(pair)

total = len(all_pairs)
print(f"── Hard filter results ────────────────────────────────────")
print(f"  Tổng input  : {total}")
print(f"  Giữ lại     : {len(kept):>5}  ({len(kept)/total*100:.1f}%)")
print(f"  Loại bỏ     : {total - len(kept):>5}  ({(total-len(kept))/total*100:.1f}%)")
print()
print(f"── Phân bổ lý do loại ─────────────────────────────────────")
for reason, items in sorted(removed.items(), key=lambda x: -len(x[1])):
    bar = "█" * (len(items) * 50 // max(1, total))
    print(f"  {reason:<18}: {len(items):>5}  ({len(items)/total*100:>5.1f}%)  {bar}")

── Hard filter results ────────────────────────────────────
  Tổng input  : 1405
  Giữ lại     :  1405  (100.0%)
  Loại bỏ     :     0  (0.0%)

── Phân bổ lý do loại ─────────────────────────────────────


## Bước 3 (tuỳ chọn): Soft filter — semantic similarity

Dùng base model `bkai-foundation-models/vietnamese-bi-encoder` để tính cosine similarity giữa query và positive. Loại các cặp có sim < `SEMANTIC_THRESHOLD` (mặc định 0.35).

**Lưu ý**: bước này tốn ~1-3 phút trên GPU, hoặc 10-15 phút trên CPU. Bật bằng `ENABLE_SEMANTIC_FILTER = True` ở cell config trên.

Bỏ qua cell này nếu chỉ cần lọc nhanh — hard filter ở bước 2 thường đã đủ.

In [5]:
if ENABLE_SEMANTIC_FILTER:
    try:
        from sentence_transformers import SentenceTransformer
        import numpy as np

        print("Loading model bkai-foundation-models/vietnamese-bi-encoder...")
        st_model = SentenceTransformer("bkai-foundation-models/vietnamese-bi-encoder")
        print(f"  Device: {st_model.device}")

        queries   = [p["query"]    for p in kept]
        positives = [p["positive"] for p in kept]

        q_emb = st_model.encode(queries,   normalize_embeddings=True, show_progress_bar=True, batch_size=64)
        p_emb = st_model.encode(positives, normalize_embeddings=True, show_progress_bar=True, batch_size=64)

        sims = (q_emb * p_emb).sum(axis=1)

        passed_semantic = []
        for pair, sim in zip(kept, sims):
            if sim >= SEMANTIC_THRESHOLD:
                pair["_qp_sim"] = float(sim)
                passed_semantic.append(pair)
            else:
                pair["_qp_sim"] = float(sim)
                removed["low_similarity"].append(pair)

        before = len(kept)
        kept   = passed_semantic
        print()
        print(f"── Semantic filter (threshold={SEMANTIC_THRESHOLD}) ──")
        print(f"  Trước : {before}")
        print(f"  Sau   : {len(kept)}")
        print(f"  Loại  : {before - len(kept)}")
        print(f"  Sim distribution:")
        print(f"    min  : {sims.min():.3f}")
        print(f"    mean : {sims.mean():.3f}")
        print(f"    median: {float(np.median(sims)):.3f}")
        print(f"    max  : {sims.max():.3f}")
    except ImportError:
        print("⚠️  Cần cài: pip install sentence-transformers")
else:
    print("Semantic filter disabled. Set ENABLE_SEMANTIC_FILTER=True để bật.")

Semantic filter disabled. Set ENABLE_SEMANTIC_FILTER=True để bật.


## Bước 4: Inspect — xem mẫu các cặp bị loại

In ra 3 mẫu mỗi loại để bạn kiểm tra xem rule có quá khắt khe không.

In [6]:
SAMPLES_PER_REASON = 3

for reason, items in sorted(removed.items(), key=lambda x: -len(x[1])):
    print(f"━━━ {reason} ({len(items)} cặp) ━━━")
    for i, pair in enumerate(items[:SAMPLES_PER_REASON]):
        sim_str = f" sim={pair['_qp_sim']:.3f}" if "_qp_sim" in pair else ""
        print(f"  [{i+1}]{sim_str} src={pair.get('_source','?')}")
        print(f"       Q: {pair['query'][:140]}")
        print(f"       P: {pair.get('positive','')[:120]}...")
    print()

## Bước 5: Dedup + tách lại train/val/test theo chương

Sau khi lọc, tách lại để có val và test set thực sự (data_generation cũ đôi khi val=0).

In [7]:
def dedup_pairs(pairs):
    """Loai bo pair trung query (normalize lowercase + collapse whitespace)."""
    seen = set()
    out = []
    for pair in pairs:
        key = __import__('hashlib').md5(
            __import__('re').sub(r'\s+', ' ', pair['query'].lower().strip()).encode()
        ).hexdigest()
        if key not in seen:
            seen.add(key)
            out.append(pair)
    return out


def split_by_chapter(pairs, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, seed=RANDOM_SEED):
    """Chia tỉ lệ train/val/test ĐỀU TRONG TỪNG CHƯƠNG để tránh data leakage
    VÀ đảm bảo val/test luôn có dữ liệu (không phụ thuộc số chương).
    """
    by_chapter = defaultdict(list)
    for p in pairs:
        by_chapter[p.get('chapter', 'unknown')].append(p)

    rng = random.Random(seed)
    train, val, test = [], [], []
    for ch, chunk in by_chapter.items():
        rng.shuffle(chunk)
        n = len(chunk)
        n_train = int(n * train_ratio)
        n_val   = int(n * val_ratio)
        # Đảm bảo ít nhất 1 mẫu vào val và test nếu chương đủ lớn
        if n >= 10:
            n_val   = max(n_val,   1)
            n_test  = max(n - n_train - n_val, 1)
            n_train = n - n_val - n_test
        else:
            n_test = n - n_train - n_val
        train.extend(chunk[:n_train])
        val.extend(chunk[n_train:n_train + n_val])
        test.extend(chunk[n_train + n_val:])
    return train, val, test


# ── Preview trên thư mục đầu tiên (SUBJECT_DIR) ────────────────────────────
before  = len(kept)
deduped = dedup_pairs(kept)
print(f'Truoc dedup : {before}')
print(f'Sau dedup   : {len(deduped)}  (loai {before - len(deduped)} cap trung)')
print()

train_set, val_set, test_set = split_by_chapter(deduped)
n = len(deduped)
print('── Split 80/10/10 theo chapter (preview: thư mục 1) ────────')
print(f'  Total : {n}')
print(f'  Train : {len(train_set):>4}  ({len(train_set)/max(1,n)*100:.1f}%)')
print(f'  Val   : {len(val_set):>4}  ({len(val_set)/max(1,n)*100:.1f}%)')
print(f'  Test  : {len(test_set):>4}  ({len(test_set)/max(1,n)*100:.1f}%)')


Truoc dedup : 1405
Sau dedup   : 1405  (loai 0 cap trung)

── Split 80/10/10 theo chapter (preview: thư mục 1) ────────
  Total : 1405
  Train : 1123  (79.9%)
  Val   :  139  (9.9%)
  Test  :  143  (10.2%)


## Buoc 6: Ghi de toan bo file JSON vao folder goc

Ghi ket qua truc tiep vao `data/training_data/` (ghi de file cu).  
**Sao luu truoc neu can giu lai ban goc.**


In [8]:
def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')


def save_mnrl(pairs, path):
    """[query, positive] cho MultipleNegativesRankingLoss."""
    with open(path, 'w', encoding='utf-8') as f:
        for p in pairs:
            f.write(json.dumps([p['query'], p['positive']], ensure_ascii=False) + '\n')


def save_triplets(pairs, path):
    """[query, positive, neg1, neg2, ...]."""
    with open(path, 'w', encoding='utf-8') as f:
        for p in pairs:
            row = [p['query'], p['positive']] + p.get('negatives', [])
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def save_json(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def process_and_save(subject_dir: Path) -> dict:
    """Lọc, chia và ghi toàn bộ file cho một thư mục subject."""
    audit_file = subject_dir / "qa_audit.jsonl"
    if not audit_file.is_file():
        print(f"  ✗ Bỏ qua (không tìm thấy qa_audit.jsonl): {subject_dir}")
        return {}

    # 1. Load
    raw_pairs = load_jsonl(audit_file)

    # 2. Hard filter
    dir_kept, dir_removed = [], defaultdict(list)
    for pair in raw_pairs:
        passed, reason = hard_filter(pair)
        if passed:
            dir_kept.append(pair)
        else:
            dir_removed[reason].append(pair)

    # 3. Semantic filter (nếu bật)
    if ENABLE_SEMANTIC_FILTER and 'st_model' in globals():
        passed_sem = []
        queries   = [p["query"]    for p in dir_kept]
        positives = [p["positive"] for p in dir_kept]
        q_emb = st_model.encode(queries,   normalize_embeddings=True, batch_size=64)
        p_emb = st_model.encode(positives, normalize_embeddings=True, batch_size=64)
        sims = (q_emb * p_emb).sum(axis=1)
        for pair, sim in zip(dir_kept, sims):
            pair["_qp_sim"] = float(sim)
            if sim >= SEMANTIC_THRESHOLD:
                passed_sem.append(pair)
            else:
                dir_removed["low_similarity"].append(pair)
        dir_kept = passed_sem

    # 4. Dedup
    deduped = dedup_pairs(dir_kept)

    # 5. Split 80/10/10 trong từng chương
    tr, va, te = split_by_chapter(deduped)

    # 6. Ghi file
    out_dir = subject_dir
    pairs_all = tr + va + te

    save_jsonl(pairs_all, out_dir / 'qa_audit.jsonl')
    save_json(tr,   out_dir / 'train_data.json')
    save_mnrl(tr,   out_dir / 'train_pairs_mnrl.jsonl')
    save_triplets(tr, out_dir / 'train_triplets.jsonl')
    save_json(va,   out_dir / 'val_data.json')
    save_mnrl(va,   out_dir / 'val_data_mnrl.jsonl')
    save_json(te,   out_dir / 'test_data.json')
    save_mnrl(te,   out_dir / 'test_data_mnrl.jsonl')

    stats_old = json.loads((out_dir / 'stats.json').read_text(encoding='utf-8')) \
        if (out_dir / 'stats.json').is_file() else {}
    stats = {
        **stats_old,
        'total_pairs': len(pairs_all),
        'train': len(tr), 'val': len(va), 'test': len(te),
        'filter_stats': {
            'input': len(raw_pairs),
            'kept': len(pairs_all),
            'removed': {r: len(items) for r, items in dir_removed.items()},
        },
        'intent_distribution': dict(Counter(p['intent_type'] for p in pairs_all)),
        'chapter_distribution': dict(Counter(str(p.get('chapter',''))[:50] for p in pairs_all)),
    }
    save_json(stats, out_dir / 'stats.json')

    report = {
        'input': {'subject': str(out_dir), 'total_pairs': len(raw_pairs)},
        'config': {
            'min_query_chars': MIN_QUERY_CHARS, 'max_query_chars': MAX_QUERY_CHARS,
            'min_positive_overlap': MIN_POSITIVE_OVERLAP,
            'semantic_filter': ENABLE_SEMANTIC_FILTER,
            'train_ratio': TRAIN_RATIO, 'val_ratio': VAL_RATIO,
        },
        'removed': {r: len(items) for r, items in dir_removed.items()},
        'output': {
            'total_kept': len(deduped),
            'reduction_pct': round((len(raw_pairs) - len(deduped)) / max(1, len(raw_pairs)) * 100, 2),
            'train': len(tr), 'val': len(va), 'test': len(te),
        },
    }
    save_json(report, out_dir / 'filter_report.json')

    return {
        'dir': str(out_dir.name),
        'input': len(raw_pairs),
        'kept': len(deduped),
        'removed_total': len(raw_pairs) - len(deduped),
        'train': len(tr), 'val': len(va), 'test': len(te),
    }


# ── Chạy pipeline cho TẤT CẢ 3 thư mục ──────────────────────────────────────
all_results = []
print('=' * 65)
print('XỬ LÝ TẤT CẢ CÁC THƯ MỤC TRAINING DATA')
print('=' * 65)

for subject_dir in SUBJECT_DIRS:
    print(f'\n▶ {subject_dir.name}  ({subject_dir.resolve()})')
    result = process_and_save(subject_dir)
    if result:
        all_results.append(result)
        n = result['kept']
        print(f'  Input  : {result["input"]} pairs')
        print(f'  Giữ lại: {result["kept"]} pairs  (loại {result["removed_total"]})')
        print(f'  train  : {result["train"]:>5}  ({result["train"]/max(1,n)*100:.1f}%)')
        print(f'  val    : {result["val"]:>5}  ({result["val"]/max(1,n)*100:.1f}%)')
        print(f'  test   : {result["test"]:>5}  ({result["test"]/max(1,n)*100:.1f}%)')
        print(f'  → Đã ghi vào {subject_dir.resolve()}')


XỬ LÝ TẤT CẢ CÁC THƯ MỤC TRAINING DATA

▶ training_data  (C:\CS431\UniPolis\data\training_data)
  Input  : 1405 pairs
  Giữ lại: 1405 pairs  (loại 0)
  train  :  1123  (79.9%)
  val    :   139  (9.9%)
  test   :   143  (10.2%)
  → Đã ghi vào C:\CS431\UniPolis\data\training_data

▶ training_data_pldc  (C:\CS431\UniPolis\data\training_data_pldc)
  Input  : 1314 pairs
  Giữ lại: 1247 pairs  (loại 67)
  train  :   990  (79.4%)
  val    :   117  (9.4%)
  test   :   140  (11.2%)
  → Đã ghi vào C:\CS431\UniPolis\data\training_data_pldc

▶ training_data_th  (C:\CS431\UniPolis\data\training_data_th\training_data_th)
  Input  : 1489 pairs
  Giữ lại: 1362 pairs  (loại 127)
  train  :  1086  (79.7%)
  val    :   132  (9.7%)
  test   :   144  (10.6%)
  → Đã ghi vào C:\CS431\UniPolis\data\training_data_th\training_data_th


## Bước 7: Tổng kết + filter_report.json

Ghi báo cáo tổng hợp tại `data/filter_report.json` để theo dõi qua nhiều lần lọc.

In [9]:
print('=' * 65)
print('DATA FILTER PIPELINE HOÀN THÀNH — TÓM TẮT TOÀN BỘ')
print('=' * 65)
print()

header = f"{'Thư mục':<28} {'Input':>6} {'Kept':>6} {'Train':>6} {'Val':>6} {'Test':>6}"
print(header)
print('-' * len(header))

total_input = total_kept = total_train = total_val = total_test = 0
for r in all_results:
    print(f"  {r['dir']:<26} {r['input']:>6} {r['kept']:>6} {r['train']:>6} {r['val']:>6} {r['test']:>6}")
    total_input += r['input']
    total_kept  += r['kept']
    total_train += r['train']
    total_val   += r['val']
    total_test  += r['test']

print('-' * len(header))
print(f"  {'TỔNG':<26} {total_input:>6} {total_kept:>6} {total_train:>6} {total_val:>6} {total_test:>6}")
print()

if total_val == 0:
    print("⚠️  val vẫn = 0! Kiểm tra lại số lượng pair trong từng chương.")
else:
    print(f"✓  Val set có dữ liệu ({total_val} pairs)")

print()
print('Filter report đã ghi vào từng thư mục (filter_report.json).')
print()
print('Bước tiếp theo:')
print('  Chạy lại fine-tuning-vietnamese-bi-encoder.ipynb')
print('  (không cần đổi đường dẫn — data vừa ghi đè vào folder cũ)')


DATA FILTER PIPELINE HOÀN THÀNH — TÓM TẮT TOÀN BỘ

Thư mục                       Input   Kept  Train    Val   Test
---------------------------------------------------------------
  training_data                1405   1405   1123    139    143
  training_data_pldc           1314   1247    990    117    140
  training_data_th             1489   1362   1086    132    144
---------------------------------------------------------------
  TỔNG                         4208   4014   3199    388    427

✓  Val set có dữ liệu (388 pairs)

Filter report đã ghi vào từng thư mục (filter_report.json).

Bước tiếp theo:
  Chạy lại fine-tuning-vietnamese-bi-encoder.ipynb
  (không cần đổi đường dẫn — data vừa ghi đè vào folder cũ)
